### Fase de Experimentación - Predicción de Default Crediticio

Este notebook documenta la fase de experimentación del proyecto final de MLOps.



### Objetivo
Predecir si un cliente de tarjeta de crédito incurrirá en default el siguiente mes.

### Dataset
UCI Credit Card Default Dataset (30,000 observaciones)

### Métrica principal
ROC-AUC para comparación de modelos.

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier

In [5]:
import sys
print(sys.executable)

c:\Users\marko\Maestria\UNI\3er semestre\mlops\Trabajo FInal\venv\Scripts\python.exe


In [6]:
RAW_PATH = "../data/raw/UCI_Credit_Card.csv"

df = pd.read_csv(RAW_PATH)
df = df.rename(columns={"default.payment.next.month": "target"})

df.shape

(30000, 25)

In [7]:
df["target"].value_counts(normalize=True)

target
0    0.7788
1    0.2212
Name: proportion, dtype: float64

Observación:
El dataset presenta desbalance moderado (~22% de defaults).
Se utilizará partición estratificada para mantener proporciones.

In [8]:
X = df.drop(columns=["target"])
y = df["target"]

# Eliminamos ID porque es identificador
if "ID" in X.columns:
    X = X.drop(columns=["ID"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((24000, 23), (6000, 23))

In [9]:
categorical_features = ["SEX", "EDUCATION", "MARRIAGE"]
numerical_features = [col for col in X_train.columns if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

Se aplica:

- Escalamiento estándar para variables numéricas.
- One-Hot Encoding para variables categóricas.
- Pipeline para garantizar reproducibilidad.

In [10]:
log_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

log_model.fit(X_train, y_train)

log_probs = log_model.predict_proba(X_test)[:, 1]
log_auc = roc_auc_score(y_test, log_probs)

log_auc

0.7100336377377392

In [11]:
log_pred = (log_probs >= 0.5).astype(int)

print("Logistic Regression AUC:", round(log_auc, 4))
print(classification_report(y_test, log_pred))

Logistic Regression AUC: 0.71
              precision    recall  f1-score   support

           0       0.82      0.97      0.89      4673
           1       0.69      0.24      0.36      1327

    accuracy                           0.81      6000
   macro avg       0.76      0.61      0.62      6000
weighted avg       0.79      0.81      0.77      6000



In [12]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=200, random_state=42))
    ]
)

rf_model.fit(X_train, y_train)

rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_probs)

rf_auc

0.7598499517260808

In [13]:
rf_pred = (rf_probs >= 0.5).astype(int)

print("Random Forest AUC:", round(rf_auc, 4))
print(classification_report(y_test, rf_pred))

Random Forest AUC: 0.7598
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      4673
           1       0.64      0.37      0.47      1327

    accuracy                           0.81      6000
   macro avg       0.74      0.66      0.68      6000
weighted avg       0.80      0.81      0.80      6000



In [18]:
xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            random_state=42,
            eval_metric="logloss"
        ))
    ]
)

xgb_model.fit(X_train, y_train)

xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_probs)

print("XGBoost AUC:", round(xgb_auc, 4))

XGBoost AUC: 0.7753


In [23]:
xgboost_pred = (rf_probs >= 0.5).astype(int)

print("XGBoost AUC:", round(xgb_auc, 4))
print(classification_report(y_test, xgboost_pred))

XGBoost AUC: 0.7753
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      4673
           1       0.64      0.37      0.47      1327

    accuracy                           0.81      6000
   macro avg       0.74      0.66      0.68      6000
weighted avg       0.80      0.81      0.80      6000



In [22]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "ROC_AUC": [log_auc, rf_auc, xgb_auc]
})

results.sort_values("ROC_AUC", ascending=False)

,Model,ROC_AUC
2,XGBoost,0.775254
1,Random Forest,0.759850
0,Logistic Regression,0.710034


In [15]:
if rf_auc > log_auc:
    champion = "Random Forest"
else:
    champion = "Logistic Regression"

champion

'Random Forest'

### Análisis Final de Modelos

Se evaluaron tres enfoques:

- Logistic Regression (modelo lineal base)
- Random Forest (ensamble tipo bagging)
- XGBoost (ensamble tipo boosting)

XGBoost obtuvo el mayor ROC-AUC (0.7753), demostrando mejor capacidad de discriminación entre clientes que incurren en default y aquellos que no.

Por lo tanto, XGBoost es seleccionado como modelo campeón para la fase de deployment.